# 🎬 AutoDub Studio — One-Click Colab Deployment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-/blob/main/Colab_Runner.ipynb)

ElevenLabs-quality **open-source automatic dubbing** — Demucs v4 · Pyannote 3.1 ·
SenseVoice-Small · CosyVoice 3.0 zero-shot cloning — wrapped in an iPhone-15
frosted-glass UI.

**How to use**
1. ▸ *Runtime* ▸ *Change runtime type* ▸ **T4 GPU** ▸ Save
2. ▸ *Runtime* ▸ **Run all**
3. When the last cell finishes, click the printed **`https://….gradio.live`** link
4. In Tab 1: upload your **audio/video master + Original SRT + Translated SRT**, hit 🔍
5. Tab 2: 🧬 matches speakers & emotions (paste a HuggingFace token in Tab 1 ▸ Advanced first — see [token guide](https://github.com/syedj7895-cell/ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-ElevenLabs-Quality-Open-Source-#-one-time-hugging-face-setup-required-for-step-3))
6. Tab 3: 🚀 renders the dubbed master mix / video

Ready to use — no build step required.

In [ ]:
# ── 0 ▸ 🩺 DIAGNOSTICS — run me FIRST (and whenever anything breaks) ──
# Prints one copy-paste block: versions · GPU · disk/RAM · HF gate access.
import importlib.metadata as md, os, shutil, sys, urllib.request

print("═" * 64)
print(" 🩺 AUTODUB DIAGNOSTICS — paste this whole block with any bug report")
print("═" * 64)
print(f"python          {sys.version.split()[0]}")
print(f"platform        {sys.platform}")

_pkgs = ["torch", "torchaudio", "torchvision", "numpy", "scipy", "librosa",
         "soundfile", "gradio", "gradio-client", "huggingface-hub",
         "demucs", "pyannote-audio", "funasr", "modelscope", "pysrt",
         "moviepy", "pydub", "ffmpeg-python", "setuptools"]
for name in _pkgs:
    try:
        print(f"{name:16s} {md.version(name)}")
    except Exception:
        print(f"{name:16s} —")

try:
    import torch
    if torch.cuda.is_available():
        pr = torch.cuda.get_device_properties(0)
        print(f"gpu              {pr.name} · {pr.total_memory / 2**30:.0f} GB · CUDA {torch.version.cuda}")
    else:
        print("gpu              NONE — CPU mode (expected during Colab GPU cooldown)")
except Exception as e:
    print("gpu              probe failed:", e)

print(f"disk  /content   {shutil.disk_usage('/content').free / 2**30:.0f} GB free")
try:
    _m = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES')
    print(f"ram              {_m / 2**30:.0f} GB")
except Exception:
    pass
print("ffmpeg           " + (shutil.which("ffmpeg") or "NOT FOUND"))

# ── HF gate probe (needs a token to be meaningful) ────────────────────
_token = (os.environ.get("HF_TOKEN") or "").strip()
if not _token and os.path.exists("/content/autodub_hf_token.txt"):
    _token = open("/content/autodub_hf_token.txt").read().strip()
if not _token:
    print("hf-gate          ⚠ no token found — paste it in Tab 1 ▸ Advanced, "
          "or set HF_TOKEN, then re-run this cell")
else:
    for repo in ("pyannote/speaker-diarization-3.1",
                 "pyannote/segmentation-3.0"):
        req = urllib.request.Request(
            f"https://huggingface.co/api/models/{repo}",
            headers={"Authorization": f"Bearer {_token}"})
        try:
            with urllib.request.urlopen(req, timeout=15) as r:
                print(f"hf-gate          ✅ {repo} accessible")
        except Exception as e:
            code = getattr(e, "code", None)
            if code in (401, 403):
                print(f"hf-gate          ❌ {repo} DENIED — open huggingface.co/{repo} "
                      "and click 'Agree and access repository' with the SAME account "
                      "that owns the token")
            else:
                print(f"hf-gate          ⚠ {repo} probe failed ({e}) — check network")
print("═" * 64)

In [ ]:
# ── 1 ▸ Runtime check + PINNED ML-stack install (torch NOT touched) ──
import importlib.metadata as md, subprocess, sys, torch

if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)} "
          f"({torch.cuda.get_device_properties(0).total_memory / 2**30:.0f} GB)")
else:
    print("⚠ No GPU on this VM — CPU mode (fine for Tabs 1–2 testing; "
          "full renders want the T4. Runtime ▸ Change runtime type ▸ T4 GPU).")

# Exact known-good pins — every fresh Colab VM resolves identically.
# torch / torchaudio / numpy are deliberately NOT here: Colab's own
# builds must stay (GPU-CUDA wheel + system compatibility).
PINNED = {
    "demucs":          "4.0.1",
    "pyannote.audio":  "3.1.1",
    "funasr":          "1.2.6",
    "modelscope":      "1.27.14",
    "pysrt":           "1.1.2",
    "pydub":           "0.25.1",
    "moviepy":         "1.0.3",
    "ffmpeg-python":   "0.2.0",
    "soundfile":       "0.13.1",
    "librosa":         "0.11.0",
}
need = []
for pkg, ver in PINNED.items():
    try:
        have = md.version(pkg)
        ok = have == ver
    except Exception:
        have, ok = None, False
    status = ("✓ " + str(have)) if ok else (str(have or "missing")
             + " -> will install " + ver)
    print(f"  {pkg:18s} {status}")
    if not ok:
        need.append(f"{pkg}=={ver}")

if need:
    print(f"\n📦 Installing {len(need)} pinned package(s) …")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *need])
    print("📦 done")

# verification asserts — fail loudly if the stack is not exactly as expected
import demucs, pyannote, pysrt, funasr, modelscope  # noqa: F401
print("✅ ML stack verified: demucs · pyannote.audio · funasr · modelscope · pysrt")

In [ ]:
# ── 2 ▸ Clone the project ─────────────────────────────────────────────
REPO_URL = ("https://github.com/syedj7895-cell/"
            "ai-video-dubber-AutoDub-Studio-Automatic-Dubbing-Engine-"
            "ElevenLabs-Quality-Open-Source-.git")
!git clone {REPO_URL} ai-video-dubber
%cd ai-video-dubber
!ls

In [ ]:
# ── realign gradio-client to gradio's EXACT pin + full-stack audit ──
import importlib.metadata as md, os, subprocess, sys
try:
    g = md.version("gradio")
    want = next((r.split("==")[1].strip() for r in md.requires("gradio")
                 if r.startswith("gradio-client")), None)
    have = md.version("gradio_client")
    if want and have != want:
        print(f"gradio {g}: gradio-client {have} -> {want}")
        subprocess.check_call([sys.executable, "-m", "pip", "install",
                               "-q", f"gradio-client=={want}"])
    else:
        print(f"gradio {g} + gradio-client {have} already aligned")
except Exception as e:
    print("client realignment skipped:", e)

# audit: every remaining unpinned requirement from the project's file
_req = os.path.join(os.getcwd(), "requirements.txt")   # we are inside the clone
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", _req])
print("✅ requirements.txt satisfied")

In [ ]:
# ── CosyVoice source-tree registration + wget-style progress bar / ETA ──
import json, os, pathlib, re, site, subprocess, sys, time

CV = pathlib.Path("/content/CosyVoice")
assert CV.is_dir() and (CV / "cosyvoice").is_dir(), \
    "Clone it first:  git clone --recursive https://github.com/FunAudioLLM/CosyVoice /content/CosyVoice"
for p in (str(CV), str(CV / "third_party" / "Matcha-TTS")):
    if p not in sys.path:
        sys.path.insert(0, p)
# persist for every future runtime process (incl. Gradio subprocesses)
pth = pathlib.Path(site.getsitepackages()[0]) / "autodub_cosyvoice.pth"
pth.write_text("\n".join([str(CV), str(CV / "third_party" / "Matcha-TTS")]), encoding="utf-8")

def _fmt_s(s: float) -> str:
    s = int(max(0, s)); m, sec = divmod(s, 60); h, m = divmod(m, 60)
    return f"{h}h{m:02d}m{sec:02d}s" if h else f"{m}m{sec:02d}s"

def _pip_count_targets(args):
    """Dry-run resolve → exact package count the real install will process."""
    try:
        r = subprocess.run([sys.executable, "-m", "pip", "install", "--dry-run",
                            "--quiet", "--report", "-", *args],
                           capture_output=True, text=True, timeout=600)
        return len(json.loads(r.stdout).get("install", []))
    except Exception:
        return None

def pip_progress(args, label="cosyvoice-reqs"):
    """pip install with a wget-style live bar + ETA countdown:
       label  62.5%[================>·] 8/11 pkgs · 214.6 MB · 9.8 MB/s · eta 1m22s
    """
    total = _pip_count_targets(args)
    if total:
        print(f"{label}: resolving done — {total} package(s) to install")
    else:
        print(f"{label}: ETA resolution unavailable — falling back to live clock")
    proc = subprocess.Popen([sys.executable, "-m", "pip",
                             "--disable-pip-version-check", "--progress-bar", "off",
                             "install", *args],
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1)
    t0, seen, mb, errors = time.time(), set(), 0.0, []
    W = 32
    def draw(frac, note):
        done = int(W * frac)
        bar = ("=" * done + ">" + "·" * max(0, W - done - 1)) if 0 < frac < 1 else "=" * W
        el = time.time() - t0
        eta = el / frac * (1 - frac) if frac > 0.02 else 0
        print(f"\r{label} {frac*100:5.1f}%[{bar}] {note} · eta {_fmt_s(eta)}   ",
              end="", flush=True)
    dl_re = re.compile(r"Downloading\s+\S+\.whl\s+\(([\d.]+)\s*(MB|kB|GB|mB)\)",
                       re.IGNORECASE)
    for line in proc.stdout:
        line = line.strip()
        m = re.match(r"Collecting\s+(\S+)", line)
        if m:
            seen.add(m.group(1).split("==")[0].split("[")[0].lower())
        dm = dl_re.search(line)
        if dm:
            mb += float(dm.group(1)) * {"kB": 0.001, "MB": 1, "mB": 1,
                                        "GB": 1024}[dm.group(2).lower()]
        if "ERROR" in line or "error:" in line:
            errors.append(line)
        if total:
            note = f"{min(len(seen), total)}/{total} pkgs"
            if mb:
                note += f" · {mb:.1f} MB · {mb / max(1e-9, time.time()-t0):.1f} MB/s"
            draw(min(0.999, len(seen) / total), note)
        elif line:
            print(f"\r{label} running · {_fmt_s(time.time()-t0)} elapsed · "
                  f"{line[:46]}   ", end="", flush=True)
    proc.wait()
    el = time.time() - t0
    if proc.returncode == 0:
        print(f"\r{label} 100.0%[{'='*W}] done · {mb:.1f} MB · in {_fmt_s(el)} total          ")
    else:
        print(f"\r{label} FAILED after {_fmt_s(el)}          ")
        for e in errors[-6:]:
            print("   ", e[:160])
        raise RuntimeError(f"pip install failed ({proc.returncode}) — see errors above")

pip_progress(["-r", str(CV / "requirements.txt")])

import cosyvoice  # sanity check
print("✅ CosyVoice importable (source-tree mode · no pip install .)")

In [ ]:
# ── 5 ▸ 🚀 LAUNCH — click the https://….gradio.live link printed below ─
!python app.py